# Estação 1 - Preparar os dados (ler CSV e explorar)


In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

df = pd.read_csv("../data/raw/morbilidade_mortalidade_hospit_fich2.csv",
                  sep=";", encoding="utf-8-sig")

df.columns = ["periodo", "regiao", "instituicao", "codigo_diagnostico_principal", "descricao_diagnostico_principal", "faixa_etaria", "sexo", "internamentos", "dias_internamentos", "ambulatorio", "obitos"]


print(df.shape)                 # (linhas, colunas)
display(df.head())              # primeiras 5 linhas
display(df.info())              # colunas, tipos e não nulos           
display(df.describe().round(2)) # estatísticas das colunas numéricas arredondado para 2 casas decimais 

(524406, 11)


,periodo,regiao,instituicao,codigo_diagnostico_principal,descricao_diagnostico_principal,faixa_etaria,sexo,internamentos,dias_internamentos,ambulatorio,obitos
0,2026-03,Região de Saúde do Norte,"Unidade Local de Saúde de Braga, E.P.E.",12,Doenças da pele e do tecido subcutâneo,[65-120[,M,2,24,0,0
1,2026-03,Região de Saúde do Algarve,"Unidade Local de Saúde do Algarve, E.P.E.",13,Doenças do aparelho osteomuscular e do tecido ...,[45-65[,M,2,3,1,0
2,2026-03,Região de Saúde do Alentejo,"Unidade Local de Saúde do Alto Alentejo, E.P.E.",13,Doenças do aparelho osteomuscular e do tecido ...,[45-65[,F,4,32,3,0
3,2026-03,Região de Saúde do Norte,"Unidade Local de Saúde do Alto Minho, E.P.E.",13,Doenças do aparelho osteomuscular e do tecido ...,[45-65[,F,1,2,8,0
4,2026-03,Região de Saúde do Norte,"Unidade Local de Saúde do Alto Minho, E.P.E.",13,Doenças do aparelho osteomuscular e do tecido ...,[65-120[,M,0,0,6,0


<class 'pandas.DataFrame'>
RangeIndex: 524406 entries, 0 to 524405
Data columns (total 11 columns):
 #   Column                           Non-Null Count   Dtype
---  ------                           --------------   -----
 0   periodo                          524406 non-null  str  
 1   regiao                           524406 non-null  str  
 2   instituicao                      524406 non-null  str  
 3   codigo_diagnostico_principal     524406 non-null  int64
 4   descricao_diagnostico_principal  524406 non-null  str  
 5   faixa_etaria                     524406 non-null  str  
 6   sexo                             524406 non-null  str  
 7   internamentos                    524406 non-null  int64
 8   dias_internamentos               524406 non-null  int64
 9   ambulatorio                      524406 non-null  int64
 10  obitos                           524406 non-null  int64
dtypes: int64(5), str(6)
memory usage: 44.0 MB


None

,codigo_diagnostico_principal,internamentos,dias_internamentos,ambulatorio,obitos
count,524406.00,524406.00,524406.00,524406.00,524406.00
mean,10.81,10.25,85.80,12.43,0.69
std,6.09,22.82,201.77,50.23,2.43
min,1.00,0.00,0.00,0.00,0.00
25%,6.00,1.00,3.00,0.00,0.00
50%,11.00,3.00,15.00,1.00,0.00
75%,15.00,8.00,71.00,5.00,0.00
max,22.00,470.00,14830.00,1267.00,120.00


# ESTAÇÃO 2 — Guardar numa base de dados e consultar com SQL

O objetivo é guardar os dados de forma estruturada numa base de dados e realizar consultas SQL para explorar a informação.


## Análise da Demora Média por Diagnóstico e Faixa Etária

Como responsável clínico, quero ver a demora média por diagnóstico e faixa etária, para perceber que perfil de doente ocupa mais dias de internamento.

In [ ]:

# Liga (ou cria) a base de dados SQLite
con = sqlite3.connect("../data/processed/morbilidade-hospital.db")

# df.to_sql escreve o DataFrame como uma TABELA dentro da base de dados.

df.to_sql("morbilidade", con, if_exists="replace", index=False)


q_diag = """
SELECT 
    codigo_diagnostico_principal,
    descricao_diagnostico_principal,
    ROUND(SUM(dias_internamentos) * 1.0 / SUM(internamentos), 2) AS demora_media
FROM morbilidade
WHERE internamentos > 0
GROUP BY 
    codigo_diagnostico_principal,
    descricao_diagnostico_principal
ORDER BY demora_media DESC;
"""
print("Demora média por diagnóstico:")
display(pd.read_sql(q_diag, con))



q = """

SELECT 

    faixa_etaria,        
    ROUND(SUM(dias_internamentos) / SUM(internamentos), 2) AS demora_media
FROM morbilidade
WHERE internamentos > 0 
GROUP BY faixa_etaria
ORDER BY demora_media DESC;
"""
print(" ")
display(pd.read_sql(q, con))

Demora média por diagnóstico.+


,codigo_diagnostico_principal,descricao_diagnostico_principal,demora_media
0,5,"Transtornos mentais, comportamentais e de neur...",23.54
1,1,Algumas doenças infecciosas e parasitárias,13.09
2,22,Códigos para fins especiais,12.39
3,19,"Lesões, envenenamento e algumas outras consequ...",11.75
4,6,Doenças do sistema nervoso,11.38
5,12,Doenças da pele e do tecido subcutâneo,10.57
6,9,Doenças do aparelho circulatório,10.09
7,3,Doenças do sangue e dos órgãos hematopoéticos ...,9.84
8,10,Doenças do aparelho respiratório,9.11
9,2,Neoplasias,8.80


,faixa_etaria,demora_media
0,[65-120[,10.0
1,[45-65[,8.0
2,[25-45[,5.0
3,[15-25[,5.0
4,[5-15[,4.0
5,[1-5[,4.0
6,[0-1[,4.0


## Conclusão

A análise da demora média por diagnóstico revela que os internamentos mais prolongados estão associados principalmente a transtornos mentais, comportamentais e de neurodesenvolvimento, com uma média de 23,54 dias por internamento. Seguem-se as doenças infeciosas e parasitárias e os códigos para fins especiais, ambos com valores superiores a 12 dias. Estes resultados sugerem que determinadas áreas clínicas apresentam maior complexidade ou necessidade de acompanhamento prolongado, refletindo-se em maiores períodos de hospitalização.


# Estação 3 — Limpeza e Transformação
## Preparação do ficheiro "Morbilidade e Mortalidade" para análise: